# Phase 2: Data Acquisition, Preprocessing & Exploratory Data Analysis
## Airline Loyalty Disengagement Project

**Author:** Sora
**Role:** Data Science Consultant
**Date:** May 2026

This notebook executes the Phase 2 workstream for the airline loyalty disengagement engagement.
It loads the four source files, applies the 12 documented cleaning decisions, constructs an
integrated customer-month panel, builds the labeled snapshot table for churn modeling, engineers
a starter set of behavioral features, and produces the EDA visualizations.

**Window definition committed in this notebook:** 6-month lookback + 6-month forward window, 2 snapshots at
January 1, 2018 and July 1, 2018. This is the final revision; earlier exploration considered
12mo+6mo and 6mo+3mo before the data justified this choice. See the Phase 2 report for the
full reasoning.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Adjust this to point at the folder containing your 4 source CSVs
DATA_DIR = "../data/raw"   # change to e.g. r"C:\\path\\to\\Airline\\Data"
OUT_DIR  = "../data/processed"   # where cleaned outputs will be saved

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 100


## 2. Load raw data
Load all four files and confirm shapes.

In [ ]:
loyalty = pd.read_csv(f"{DATA_DIR}/Customer Loyalty History.csv", encoding="utf-8-sig")
flight  = pd.read_csv(f"{DATA_DIR}/Customer Flight Activity.csv", encoding="utf-8-sig")
calendar = pd.read_csv(f"{DATA_DIR}/Calendar.csv")
data_dict = pd.read_csv(f"{DATA_DIR}/Airline Loyalty Data Dictionary.csv")

print(f"Loyalty History : {loyalty.shape}")
print(f"Flight Activity : {flight.shape}")
print(f"Calendar        : {calendar.shape}")
print(f"Data Dictionary : {data_dict.shape}")


## 3. Initial Audit

We examine each file for missingness, duplicate keys, dtype issues, and impossible values
before any cleaning. The findings below drive the cleaning decisions in section 4.

In [ ]:
# Loyalty audit
print("=== LOYALTY HISTORY ===")
print(f"Shape: {loyalty.shape}")
print(f"\nNulls per column:\n{loyalty.isnull().sum()}")
print(f"\nDuplicate Loyalty Numbers: {loyalty['Loyalty Number'].duplicated().sum()}")
print(f"\nCountries: {loyalty['Country'].unique()}")
print(f"\nSalary stats:\n{loyalty['Salary'].describe()}")
print(f"\nSalary < 0 count: {(loyalty['Salary'] < 0).sum()}")
print(f"\nCancelled members: {loyalty['Cancellation Year'].notna().sum()} ({100*loyalty['Cancellation Year'].notna().mean():.1f}%)")


In [ ]:
# Flight audit
print("=== FLIGHT ACTIVITY ===")
print(f"Shape: {flight.shape}")
print(f"Year range: {flight['Year'].min()}-{flight['Year'].max()}")
print(f"Unique members in flight file: {flight['Loyalty Number'].nunique()}")
print(f"\nExact duplicate rows: {flight.duplicated().sum()}")
print(f"(LoyaltyNo, Year, Month) key duplicates: {flight.duplicated(subset=['Loyalty Number','Year','Month']).sum()}")
print(f"\nNegative values in any numeric column:")
for c in ['Total Flights','Distance','Points Accumulated','Points Redeemed','Dollar Cost Points Redeemed']:
    print(f"  {c}: {(flight[c]<0).sum()}")


### Audit findings

1. **Flight data only covers 2017–2018**, not 2012–2018 as the brief implied. This constrains our snapshot design.
2. **4,238 missing salaries** — exactly equals the count of College-educated members (100% systematic missingness — confirm below).
3. **20 negative-salary rows** (impossible values).
4. **1,922 exact duplicate flight rows** + ~1,949 key-duplicate rows with different values.
5. **53.7% of flight rows are zero-activity months** — by design (monthly panel logs all members).
6. **All 16,737 loyalty members appear in the flight file** (100% referential integrity).

In [ ]:
# Confirm salary/education systematic missingness
print("Salary missing × Education cross-tab:")
print(pd.crosstab(loyalty['Salary'].isna(), loyalty['Education'], margins=True))
# 100% of missing-salary rows are College-educated


### Tenure-at-cancellation anomaly: the 8-month spike

A separate diagnostic revealed that **1,048 of 2,067 cancelled members (50.7%) cancelled at exactly
8 months tenure**, and every one of them has `cancellation_month = enrollment_month + 8 (mod 12)`.
This is not customer behavior — it is a system rule firing (likely an 8-month auto-cancellation
on the "2018 Promotion" and a similar rule on Standard members).

**Implication:** the formal Cancellation field is heavily contaminated; do not use it as the
churn target. Behavioral churn (defined below) is the only valid label.

In [ ]:
# Reproduce the 8-month spike for reference
cancelled = loyalty.dropna(subset=['Cancellation Year']).copy()
cancelled['enroll_date'] = pd.to_datetime(cancelled[['Enrollment Year','Enrollment Month']].rename(
    columns={'Enrollment Year':'year','Enrollment Month':'month'}).assign(day=1))
cancelled['cancel_date'] = pd.to_datetime(cancelled[['Cancellation Year','Cancellation Month']].rename(
    columns={'Cancellation Year':'year','Cancellation Month':'month'}).astype(int).assign(day=1))
cancelled['tenure_months'] = ((cancelled['cancel_date'] - cancelled['enroll_date']).dt.days/30.44).round(0).astype(int)
print(cancelled['tenure_months'].value_counts().sort_index().head(15))


## 4. Apply Cleaning Decisions

Twelve decisions documented in the Phase 2 report. Each one is applied below with the reasoning recapped briefly.

In [ ]:
# Decisions 1 & 2: salary cleaning
# Negative salaries → NaN; build Salary_Band with Unknown for College (systematic missingness)
loyalty.loc[loyalty['Salary'] < 0, 'Salary'] = np.nan
q1, q3 = loyalty['Salary'].dropna().quantile([0.25, 0.75]).values
def salary_band(s):
    if pd.isna(s): return "Unknown"
    if s < q1: return "Low"
    if s < q3: return "Mid"
    return "High"
loyalty['Salary_Band'] = loyalty['Salary'].apply(salary_band)

# Build enrollment and cancellation dates
loyalty['enroll_date'] = pd.to_datetime(loyalty[['Enrollment Year','Enrollment Month']].rename(
    columns={'Enrollment Year':'year','Enrollment Month':'month'}).assign(day=1))
loyalty['cancel_date'] = pd.NaT
mask = loyalty['Cancellation Year'].notna()
loyalty.loc[mask, 'cancel_date'] = pd.to_datetime(
    loyalty.loc[mask, ['Cancellation Year','Cancellation Month']].rename(
        columns={'Cancellation Year':'year','Cancellation Month':'month'}).astype(int).assign(day=1))

print(f"Salary_Band distribution:\n{loyalty['Salary_Band'].value_counts()}")


In [ ]:
# Decisions 3 & 4: deduplicate flights
print(f"Raw flight rows: {len(flight):,}")
flight = flight.drop_duplicates()
print(f"After dropping exact duplicates: {len(flight):,}")
flight = flight.groupby(['Loyalty Number','Year','Month'], as_index=False).sum()
print(f"After aggregating key duplicates: {len(flight):,}")
flight['flight_date'] = pd.to_datetime(flight[['Year','Month']].assign(day=1))


In [ ]:
# Decisions 5 & 6: drop flights outside member's enrollment-to-cancellation window
flight = flight.merge(loyalty[['Loyalty Number','enroll_date','cancel_date']], on='Loyalty Number')
before = len(flight)
flight = flight[flight['flight_date'] >= flight['enroll_date']]
print(f"Dropped {before - len(flight):,} pre-enrollment flight rows")
before = len(flight)
flight = flight[~(flight['cancel_date'].notna() & (flight['flight_date'] > flight['cancel_date']))]
print(f"Dropped {before - len(flight):,} post-cancellation flight rows")
print(f"\nFinal cleaned panel: {flight.shape}")
print(f"Members covered: {flight['Loyalty Number'].nunique()}")


## 5. Save cleaned artifacts

In [ ]:
flight.to_csv(f"{OUT_DIR}/panel_cleaned.csv", index=False)
loyalty.to_csv(f"{OUT_DIR}/loyalty_cleaned.csv", index=False)
print("Saved panel_cleaned.csv and loyalty_cleaned.csv")


## 6. Build labeled snapshots

**Window definition (committed):**
- Lookback: 6 months before each as-of date
- Forward: 6 months after each as-of date
- Eligibility: member must have ≥1 flight in lookback AND have been enrolled at least 6 months
- Snapshots: Jan 1 2018 and Jul 1 2018 (spaced 6 months apart, no forward-window overlap)

**Churn label:** A member is churned at an as-of date T if and only if they had ≥1 flight in the prior 6
months AND 0 flights in the next 6 months. This is behavioral, not contractual.

In [ ]:
SNAPSHOTS = [pd.Timestamp("2018-01-01"), pd.Timestamp("2018-07-01")]
LB, FWD = 6, 6

labels = []
for asof in SNAPSHOTS:
    elig = loyalty[loyalty['enroll_date'] <= asof - pd.DateOffset(months=LB)].copy()
    elig = elig[(elig['cancel_date'].isna()) | (elig['cancel_date'] > asof)]
    lb_f  = flight[(flight['flight_date'] >= asof - pd.DateOffset(months=LB)) & (flight['flight_date'] < asof)]
    fwd_f = flight[(flight['flight_date'] >= asof) & (flight['flight_date'] < asof + pd.DateOffset(months=FWD))]
    lb_flights  = lb_f.groupby('Loyalty Number')['Total Flights'].sum()
    fwd_flights = fwd_f.groupby('Loyalty Number')['Total Flights'].sum()
    elig['lb_flights']  = elig['Loyalty Number'].map(lb_flights).fillna(0)
    elig['fwd_flights'] = elig['Loyalty Number'].map(fwd_flights).fillna(0)
    elig = elig[elig['lb_flights'] > 0]
    elig['as_of_date'] = asof
    elig['churned'] = (elig['fwd_flights'] == 0).astype(int)
    labels.append(elig[['Loyalty Number','as_of_date','lb_flights','fwd_flights','churned']])
    print(f"Snapshot {asof.date()}: eligible={len(elig):,}, churners={int(elig['churned'].sum())}, rate={elig['churned'].mean()*100:.2f}%")

labeled = pd.concat(labels, ignore_index=True)
print(f"\nTotal labeled rows: {len(labeled):,}")
print(f"Unique members: {labeled['Loyalty Number'].nunique():,}")
print(f"Overall churn rate: {labeled['churned'].mean()*100:.2f}%")
labeled.to_csv(f"{OUT_DIR}/labeled_snapshots.csv", index=False)


## 7. Engineer starter features

We build a starter set of behavioral features at each snapshot. These respect the "as-of"
discipline — all features computed using only data with timestamps strictly before the as-of date.
This is the foundation for the Phase 3 feature set.

In [ ]:
feature_rows = []
for asof in SNAPSHOTS:
    lb_start = asof - pd.DateOffset(months=LB)
    lb_mid   = asof - pd.DateOffset(months=LB//2)
    lb_data  = flight[(flight['flight_date'] >= lb_start) & (flight['flight_date'] < asof)].copy()

    agg = lb_data.groupby('Loyalty Number').agg(
        lb_flights=('Total Flights','sum'),
        lb_distance=('Distance','sum'),
        lb_pts_acc=('Points Accumulated','sum'),
        lb_pts_red=('Points Redeemed','sum'),
        lb_months_with_flights=('Total Flights', lambda s: (s>0).sum()),
    ).reset_index()

    flying = lb_data[lb_data['Total Flights']>0]
    last_flight = flying.groupby('Loyalty Number')['flight_date'].max()
    agg['lb_last_flight']    = agg['Loyalty Number'].map(last_flight)
    agg['lb_recency_months'] = ((asof - agg['lb_last_flight']).dt.days/30.44).round(1)

    first_half  = lb_data[lb_data['flight_date'] < lb_mid].groupby('Loyalty Number')['Total Flights'].sum()
    second_half = lb_data[lb_data['flight_date'] >= lb_mid].groupby('Loyalty Number')['Total Flights'].sum()
    agg['first_half_flights']  = agg['Loyalty Number'].map(first_half).fillna(0)
    agg['second_half_flights'] = agg['Loyalty Number'].map(second_half).fillna(0)
    agg['lb_trend_diff']       = agg['second_half_flights'] - agg['first_half_flights']
    agg['lb_redemption_ratio'] = np.where(agg['lb_pts_acc']>0, agg['lb_pts_red']/agg['lb_pts_acc'], 0)
    agg['as_of_date'] = asof
    feature_rows.append(agg)

feats = pd.concat(feature_rows, ignore_index=True)
df = labeled.drop(columns=['lb_flights','fwd_flights']).merge(feats, on=['Loyalty Number','as_of_date'], how='left')
df = df.merge(loyalty[['Loyalty Number','Gender','Education','Salary_Band','Marital Status','Loyalty Card','Province','enroll_date']],
              on='Loyalty Number')
df['tenure_months'] = ((df['as_of_date'] - df['enroll_date']).dt.days/30.44).round(1)
df.to_csv(f"{OUT_DIR}/labeled_features.csv", index=False)
print(f"Feature table: {df.shape}")
print(f"Columns: {list(df.columns)}")


## 8. EDA — Demographics overview

Member population is roughly balanced on gender, dominated by Bachelor education
(with College as the systematically missing-salary group), Married marital status, and the
Star tier in the loyalty pyramid.

In [ ]:
TIER_ORDER = ['Star','Nova','Aurora']
PALETTE = ['#1F3A5F','#2E5C8A','#4A90C2','#7FB3D3','#A8C8E0','#D4A017']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Member Demographics Overview ({len(loyalty):,} members)', fontsize=13, fontweight='bold')

loyalty['Gender'].value_counts().plot(kind='bar', ax=axes[0,0], color=PALETTE[:2], rot=0)
axes[0,0].set_title('Gender'); axes[0,0].set_ylabel('Members'); axes[0,0].set_xlabel('')

order_edu = ['High School or Below','College','Bachelor','Master','Doctor']
loyalty['Education'].value_counts().reindex(order_edu).plot(kind='bar', ax=axes[0,1], color=PALETTE[:5], rot=30)
axes[0,1].set_title('Education (College has missing salary)'); axes[0,1].set_xlabel('')

loyalty['Marital Status'].value_counts().plot(kind='bar', ax=axes[0,2], color=PALETTE[:3], rot=0)
axes[0,2].set_title('Marital Status'); axes[0,2].set_xlabel('')

loyalty['Loyalty Card'].value_counts().reindex(TIER_ORDER).plot(kind='bar', ax=axes[1,0],
    color=[{'Star':'#1F3A5F','Nova':'#2E5C8A','Aurora':'#D4A017'}[t] for t in TIER_ORDER], rot=0)
axes[1,0].set_title('Loyalty Card Tier'); axes[1,0].set_xlabel('')

loyalty['Salary_Band'].value_counts().reindex(['Low','Mid','High','Unknown']).plot(
    kind='bar', ax=axes[1,1], color=PALETTE[:4], rot=0)
axes[1,1].set_title('Salary Band'); axes[1,1].set_xlabel('')

loyalty['Province'].value_counts().head(8).plot(kind='barh', ax=axes[1,2], color=PALETTE[1])
axes[1,2].set_title('Province (Top 8)'); axes[1,2].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_demographics.png", dpi=110, bbox_inches='tight')
plt.show()


## 9. EDA — Flying behavior

Member-level annual flight distribution is right-skewed (median 16/year). Seasonality is strong
(2.4x peak-to-trough), with July as the peak and February as the trough.

**Key finding to flag:** tier does NOT predict flying frequency. Star, Nova, and Aurora members
all have median ~18 annual flights with nearly identical distributions. Tier in this dataset
appears to be assigned at enrollment, not earned through activity.

In [ ]:
annual = flight.groupby(['Loyalty Number','Year'])['Total Flights'].sum().reset_index()
tier_lookup = loyalty.set_index('Loyalty Number')['Loyalty Card']
annual['Tier'] = annual['Loyalty Number'].map(tier_lookup)
TIER_COLORS = {'Star':'#1F3A5F','Nova':'#2E5C8A','Aurora':'#D4A017'}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Flying Behavior', fontsize=13, fontweight='bold')

axes[0,0].hist(annual['Total Flights'], bins=range(0,80,2), color=PALETTE[1], edgecolor='white')
axes[0,0].set_title('Annual flights per member (log scale)')
axes[0,0].set_xlabel('Flights per year'); axes[0,0].set_ylabel('# member-years'); axes[0,0].set_yscale('log')

data_by_tier = [annual[annual['Tier']==t]['Total Flights'].values for t in TIER_ORDER]
bp = axes[0,1].boxplot(data_by_tier, tick_labels=TIER_ORDER, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], [TIER_COLORS[t] for t in TIER_ORDER]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0,1].set_title('Annual flights by tier (tier does NOT predict behavior)')
axes[0,1].set_ylabel('Flights per year')

seasonality = flight.groupby('Month')['Total Flights'].mean()
axes[1,0].bar(seasonality.index, seasonality.values, color=PALETTE[1])
axes[1,0].set_title('Seasonality: avg flights per member-month')
axes[1,0].set_xlabel('Month'); axes[1,0].set_xticks(range(1,13))

monthly_total = flight.groupby('flight_date')['Total Flights'].sum()
axes[1,1].plot(monthly_total.index, monthly_total.values, color=PALETTE[0], linewidth=2)
axes[1,1].set_title('Aggregate flights per month (2017-2018)')
axes[1,1].set_xlabel('Month'); axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_behavior.png", dpi=110, bbox_inches='tight')
plt.show()


## 10. EDA — Churn signals

Now we examine which features actually discriminate churned vs. non-churned members.

**Key demographic finding:** Aurora (the top tier) has the highest churn rate at **0.88%** (1.3x baseline)
— substantially above Nova (0.63%) and Star (0.62%). This is striking and aligns with the
brief's hypothesis that "high-value customers may stop flying without being noticed." Given
that tier does not track behavior in this data, an Aurora member who disengages produces
no automatic warning signal — exactly the silent attrition the airline is trying to detect.

Other demographic splits are narrow (max-to-min ratios of 1.2–1.5x):
Female 0.73% vs Male 0.62%; Married 0.72% vs Divorced 0.56%; Salary_Band Low 0.72% vs High 0.60%.
Education shows a wider spread (Master 0.85% vs Doctor 0.29%), but small subgroup sizes for
Master and Doctor make this less reliable. The model will still rely primarily on behavioral
RFM features, with tier (and especially the Aurora indicator) providing useful context.

The four most discriminative behavioral features at the 6mo+6mo label:
- **Months with flights:** 1 active month = 1.5% churn vs 6 active months = 0.17% (9x signal)
- **Lookback flights:** 1 flight = 2.4% vs 21+ = 0.5% (4.7x signal)
- **Recency:** 5+ months since last flight = 3.4% vs 1 month = 0.6% (5.5x signal at the tail)
- **Trend:** modest signal — steeply declining members slightly more likely to churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f'Strongest churn discriminators (baseline {df["churned"].mean()*100:.2f}%)', fontsize=13, fontweight='bold')

ax = axes[0,0]
df['lb_flights_bucket'] = pd.cut(df['lb_flights'], bins=[0,1,3,5,10,20,200], labels=['1','2-3','4-5','6-10','11-20','21+'])
g = df.groupby('lb_flights_bucket', observed=True)['churned'].agg(['mean','size'])
g['mean'].plot(kind='bar', ax=ax, color=PALETTE[1], rot=0)
ax.axhline(df['churned'].mean(), color='red', linestyle='--', alpha=0.5)
ax.set_title('Churn rate by lookback flights bucket')
ax.set_xlabel('Flights in lookback (6mo)'); ax.set_ylabel('Churn rate')
for i, (v, n) in enumerate(zip(g['mean'].values, g['size'].values)):
    ax.text(i, v+0.0003, f"{v*100:.2f}%\nn={int(n):,}", ha='center', fontsize=8)

ax = axes[0,1]
g = df.groupby('lb_months_with_flights')['churned'].agg(['mean','size'])
g['mean'].plot(kind='bar', ax=ax, color=PALETTE[1], rot=0)
ax.axhline(df['churned'].mean(), color='red', linestyle='--', alpha=0.5)
ax.set_title('Churn rate by months-with-flights in lookback')
ax.set_xlabel('Months with at least 1 flight (of 6)'); ax.set_ylabel('Churn rate')
for i, (v, n) in enumerate(zip(g['mean'].values, g['size'].values)):
    ax.text(i, v+0.0003, f"{v*100:.2f}%\nn={int(n):,}", ha='center', fontsize=8)

ax = axes[1,0]
df['recency_bucket'] = pd.cut(df['lb_recency_months'], bins=[-0.1,0.5,1.5,2.5,3.5,4.5,7], labels=['0','1','2','3','4','5+'])
g = df.groupby('recency_bucket', observed=True)['churned'].agg(['mean','size'])
g['mean'].plot(kind='bar', ax=ax, color=PALETTE[1], rot=0)
ax.axhline(df['churned'].mean(), color='red', linestyle='--', alpha=0.5)
ax.set_title('Churn rate by months-since-last-flight')
ax.set_xlabel('Months since last flight'); ax.set_ylabel('Churn rate')
for i, (v, n) in enumerate(zip(g['mean'].values, g['size'].values)):
    ax.text(i, v+0.0003, f"{v*100:.2f}%\nn={int(n):,}", ha='center', fontsize=8)

ax = axes[1,1]
df['trend_bucket'] = pd.cut(df['lb_trend_diff'], bins=[-200,-5,-2,-0.1,0.1,2,5,200], labels=['<=-5','-4 to -2','-1','0','+1','+2 to +4','>=5'])
g = df.groupby('trend_bucket', observed=True)['churned'].agg(['mean','size'])
g['mean'].plot(kind='bar', ax=ax, color=PALETTE[1], rot=0)
ax.axhline(df['churned'].mean(), color='red', linestyle='--', alpha=0.5)
ax.set_title('Churn rate by trend (2nd half − 1st half flights)')
ax.set_xlabel('Trend (positive = accelerating)'); ax.set_ylabel('Churn rate')
for i, (v, n) in enumerate(zip(g['mean'].values, g['size'].values)):
    ax.text(i, v+0.0003, f"{v*100:.2f}%\nn={int(n):,}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_churn_discriminators.png", dpi=110, bbox_inches='tight')
plt.show()


## 11. Phase 2 Summary of Findings

**Data quality**
- Flight data spans 2017–2018 only (not 2012–2018 as the brief implied)
- Salary missingness (25.3%) is 100% systematic — entirely the College education group
- Cancellation field is contaminated: ~50% of cancellations are an 8-month auto-cancellation system rule, not user-initiated churn

**Window decision (revised twice based on evidence)**
- Final: 6-month lookback + 6-month forward window
- 2 snapshots: Jan 1, 2018 and Jul 1, 2018
- 0.68% overall churn rate (154 churners across 22,759 labeled rows) — severe imbalance requires aggressive class weighting

**Behavioral patterns to carry into Phase 3**
- Tier does NOT track flying behavior — cannot rely on it as a primary segmentation axis
- BUT Aurora (top tier) has notably higher churn than Star/Nova (0.88% vs 0.62%) — high-value silent attrition is real
- Other demographics are weak churn predictors (max-to-min ratios < 1.5x apart from Education's noisier signal)
- Behavioral features show strong discrimination at 6mo+6mo labels:
  - Months active: 9x signal
  - Recency at tail (5+ months): 5.5x signal
  - Lookback flights volume: 4.7x signal
  - Trend (decline): 1.7x signal

**Implications for Phase 3**
- LightGBM primary, logistic regression baseline as planned
- `scale_pos_weight ≈ 147` to handle 0.68% churn rate
- Evaluate primarily on PR-AUC; ROC-AUC alone is misleading at extreme imbalance
- Segmentation must rely on RFM behavioral features; tier and demographics will provide context, not structure

## 12. Final saved artifacts

The following files have been written to your output directory and will be inputs to Phase 3.

In [ ]:
import os
for fname in ['panel_cleaned.csv','loyalty_cleaned.csv','labeled_snapshots.csv','labeled_features.csv',
              'eda_demographics.png','eda_behavior.png','eda_churn_discriminators.png']:
    path = os.path.join(OUT_DIR, fname)
    if os.path.exists(path):
        print(f"  {fname}: {os.path.getsize(path):,} bytes")
